# Web Scraping

## Billboard Scraping

The first step is to scrape the Billboard Global 200 chart for a given week.

In [202]:
import time
from operator import contains

#Import packages
import requests
import lxml.html as lx
import re
import pandas as pd
from datetime import datetime
import sqlite3 as sql

In [203]:
#Set up base url and headers for scraping
base_url = 'https://www.billboard.com/charts'
headers = {
    'User-Agent': "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/26.0.1 Safari/605.1.15"
}

In [204]:
def clean_string(stc):
    new_string = re.findall(r'(\S.*\S)', stc)[0]
    return new_string

In [205]:
def clean_int(stc):
    new_int = re.sub(r'\s*', '', stc)
    return new_int

In [206]:
def clean_date(stc):
    str_date = re.findall(r"(?<=of )(.*)(?=\s)", stc)[0]
    new_date_format = datetime.strptime(str_date, "%B %d, %Y").date()
    reformatted_str_date = new_date_format.strftime("%m-%d-%Y")
    return reformatted_str_date

In [207]:
def get_billboard_data(billboard_html, row):

    billboard_row = {}
    billboard_row['rank'] = row
    billboard_row['song_name'] = clean_string(billboard_html.xpath(f"//ul[@data-detail-target='{row}']//h3/text()")[0])

    len_artist = len(billboard_html.xpath(f"//ul[@data-detail-target='{row}']//a"))
    if len_artist == 0:
        billboard_row['artist'] = clean_string(billboard_html.xpath(f"//ul[@data-detail-target='{row}']//li[@class='o-chart-results-list__item // lrv-u-flex-grow-1 lrv-u-flex lrv-u-flex-direction-column lrv-u-justify-content-center lrv-u-padding-l-050 lrv-u-padding-l-00@mobile-max u-max-width-397']//span/text()")[0])
        billboard_row['artist_link'] = None
    else:
        artists = billboard_html.xpath(f"//ul[@data-detail-target='{row}']//a/text()")[0]
        links = billboard_html.xpath(f"//ul[@data-detail-target='{row}']//a/@href")[0]
        for i in range(1, len_artist):
            artists = artists + ', ' + billboard_html.xpath(f"//ul[@data-detail-target='{row}']//a/text()")[i]
            links = links + ', ' + billboard_html.xpath(f"//ul[@data-detail-target='{row}']//a/@href")[i]
        billboard_row['artist'] = artists
        billboard_row['artist_link'] = links
    billboard_row['last_week_rank'] = clean_int(billboard_html.xpath(f"//ul[@data-detail-target='{row}']//div[@class='lrv-u-flex lrv-u-justify-content-space-between lrv-u-align-items-center u-grid-gap-125 u-align-self-stretch u-margin-b-0.375']//li//span")[0].text_content())
    billboard_row['peak'] = clean_int(billboard_html.xpath(f"//ul[@data-detail-target='{row}']//div[@class='lrv-u-flex lrv-u-justify-content-space-between lrv-u-align-items-center u-grid-gap-125 u-align-self-stretch u-margin-b-0.375']//li//span")[1].text_content())
    billboard_row['weeks_chart'] = clean_int(billboard_html.xpath(f"//ul[@data-detail-target='{row}']//div[@class='lrv-u-flex lrv-u-justify-content-space-between lrv-u-align-items-center u-grid-gap-125 u-align-self-stretch u-margin-b-0.375']//li//span")[2].text_content())
    billboard_row['week'] = clean_date(billboard_html.xpath("//div[@class='charts-title lrv-u-position-relative // lrv-u-width-100p u-padding-t-2.25 u-margin-b-0.625@tablet lrv-u-padding-t-050@mobile-max']//div[@class = 'u-padding-a-1@mobile-max u-padding-b-0.375@mobile-max']")[0].text_content())

    row_df = pd.DataFrame(billboard_row, index = [billboard_row["rank"]])
    return row_df

In [208]:
def get_html(link):
    billboard_info = requests.get(link, headers = headers)
    billboard_info.raise_for_status()
    billboard_html = lx.fromstring(billboard_info.text)
    return billboard_html

In [209]:
def get_billboard_chart(link):

    billboard_rows = []

    billboard_html = get_html(link)
    num_of_elements = len(billboard_html.xpath("//div[@class='o-chart-results-list-row-container']"))

    for i in range(1, num_of_elements + 1):
        new_row = get_billboard_data(billboard_html, i)
        billboard_rows.append(new_row)

    complete_chart = pd.concat(billboard_rows)
    complete_chart_indexed = complete_chart.set_index("rank", drop=True)

    return complete_chart_indexed

In [210]:
def get_billboard_chart_links(link):
    billboard_html = get_html(link)
    base_direc = 'https://www.billboard.com'

    charts_dic = {}
    chart_categories = billboard_html.xpath("//ul[@class='lrv-a-unstyle-list lrv-u-background-color-black lrv-u-color-white u-padding-b-3.125']//ul[@aria-labelledby='charts-menu-item-hits-of-the-world']//a/text()")
    chart_links = billboard_html.xpath("//ul[@class='lrv-a-unstyle-list lrv-u-background-color-black lrv-u-color-white u-padding-b-3.125']//ul[@aria-labelledby='charts-menu-item-hits-of-the-world']//a/@href")

    #US table
    hot_100_link = chart_links.append(billboard_html.xpath("//ul[@class='lrv-a-unstyle-list lrv-u-background-color-black lrv-u-color-white u-padding-b-3.125']//ul[@aria-labelledby='charts-menu-item-top-charts']//a/@href")[0])
    chart_categories.append('U.S. Hot 100 Songs')

    #Global 200
    hot_200_link = chart_links.append(billboard_html.xpath("//ul[@class='lrv-a-unstyle-list lrv-u-background-color-black lrv-u-color-white u-padding-b-3.125']//ul[@aria-labelledby='charts-menu-item-global']//a/@href")[0])
    chart_categories.append('Worldwide Top 200 Songs')

    for element, element_link in zip(chart_categories, chart_links):
        charts_dic[clean_string(element)] = base_direc + element_link

    return charts_dic

In [211]:
def get_all_billboard_global_charts(url):
    world_chart_links = get_billboard_chart_links(url)
    adjusted_world_charts = [chart_name for chart_name in list(world_chart_links.keys()) if re.search("Artist|Album|Global|Top Philippine|Thai Country|Vietnamese|Singles", chart_name) is None]

    all_charts = {}

    for chart in adjusted_world_charts:
        time.sleep(1)
        print(chart)
        chart_link = world_chart_links[chart]
        chart_table = get_billboard_chart(chart_link)
        column_add = [chart] * chart_table.shape[0]
        chart_table.insert(loc = chart_table.shape[1], column = "chart", value = column_add)
        all_charts[chart] = chart_table

    return all_charts

In [102]:
all_charts_list_df = get_all_billboard_global_charts(base_url)

Billboard Arabic Hot 100
Billboard Argentina Hot 100
Billboard Brasil Hot 100
Billboard Colombia Hot 100
Billboard Canadian Hot 100
Billboard Italy Hot 100
Billboard Japan Hot 100
Billboard Korea Hot 100
Billboard Philippines Hot 100
Billboard Thailand Top Thai Songs
Billboard Vietnam Hot 100
Australia Songs
Austria Songs
Belgium Songs
Bolivia Songs
Chile Songs
China TME UNI Chart
Croatia Songs
Czech Republic Songs
Denmark Songs
Ecuador Songs
Finland Songs
France Songs
Germany Songs
Greece Songs
Hong Kong Songs
Hungary Songs
Iceland Songs
India Songs
Indonesia Songs
Ireland Songs
Luxembourg Songs
Malaysia Songs
Mexico Songs
Netherlands Songs
New Zealand Songs
Norway Songs
Peru Songs
Poland Songs
Portugal Songs
Romania Songs
Singapore Songs
Slovakia Songs
South Africa Songs
Spain Songs
Sweden Songs
Switzerland Songs
Taiwan Songs
Turkey Songs
U.K. Songs
U.S. Hot 100 Songs
Worldwide Top 200 Songs


In [103]:
all_charts_df = pd.concat(all_charts_list_df).reset_index(drop = True)

In [109]:
all_charts_df

,song_name,artist,artist_link,last_week_rank,peak,weeks_chart,Week,Chart
0,FA9LA,Flipperachi,None,1,1,10,02-28-2026,Billboard Arabic Hot 100
1,Yama,DYSTINCT,None,2,1,17,02-28-2026,Billboard Arabic Hot 100
2,Ma Tnsani,Vanco Featuring AYA,None,3,2,29,02-28-2026,Billboard Arabic Hot 100
3,Msh Awl Marra,"Lege-Cy, Ghaliaa",None,7,4,3,02-28-2026,Billboard Arabic Hot 100
4,Ta3al,DYSTINCT,None,4,3,4,02-28-2026,Billboard Arabic Hot 100
...,...,...,...,...,...,...,...,...
1645,Danza Kuduro,Don Omar,https://www.billboard.com/artist/don-omar/,-,140,37,02-28-2026,Worldwide Top 200 Songs
1646,Who,Jimin,https://www.billboard.com/artist/jimin/,-,1,74,02-28-2026,Worldwide Top 200 Songs
1647,Too Sweet,Hozier,https://www.billboard.com/artist/hozier/,-,1,98,02-28-2026,Worldwide Top 200 Songs
1648,Hips Don't Lie,"Shakira, Wyclef Jean","https://www.billboard.com/artist/shakira/, htt...",-,134,20,02-28-2026,Worldwide Top 200 Songs


In [198]:
#Total number of mentions of artist in charts
total_appearances_all_charts = all_charts_df.groupby("artist").count().sort_values(by=['song_name'], ascending = False)[['song_name', 'week']]
total_appearances_all_charts['week'] = all_charts_df.loc[0, 'week']
total_appearances_all_charts  = total_appearances_all_charts.rename(columns = {"song_name" : "count"}).reset_index()
total_appearances_all_charts

,artist,count,week
0,Bad Bunny,180,02-28-2026
1,Olivia Dean,62,02-28-2026
2,sombr,42,02-28-2026
3,Taylor Swift,37,02-28-2026
4,Djo,25,02-28-2026
...,...,...,...
696,Jade LeMac,1,02-28-2026
697,Jamie Fine,1,02-28-2026
698,Jane Zhang,1,02-28-2026
699,Jason Aldean,1,02-28-2026


In [197]:
#How many charts each artist is charting in
unique_chart_appearances = all_charts_df.groupby(by = ["artist", "chart"]).count().groupby("artist").count().iloc[:, 0].sort_values(ascending=False).to_frame()
unique_chart_appearances['week'] = all_charts_df.loc[0, 'week']
unique_chart_appearances = unique_chart_appearances.rename(columns = {"song_name" : "unique_count"}).reset_index()
unique_chart_appearances

,artist,unique_count,week
0,Bad Bunny,39,02-28-2026
1,Taylor Swift,28,02-28-2026
2,Djo,25,02-28-2026
3,"Dave, Tems",24,02-28-2026
4,RAYE,24,02-28-2026
...,...,...,...
696,"J. Cole, Future",1,02-28-2026
697,"J. Cole, Tems, Future",1,02-28-2026
698,JC Reyes & Slayter,1,02-28-2026
699,"Jackie Chan, Emil Wakin Chau, Leehom Wang, Nic...",1,02-28-2026


## Save tables to Database

In [199]:
#Save as sql file
# Import/Create DB from CSV
csv_file_path = '.../Final_Project/all_billboard_carts.csv'
sqlite_db_path = '../Final_Project/data.db' # Name of the table to be created in SQLite

conn = sql.connect(sqlite_db_path) # connect to a NEW database!

In [200]:
table_name_charts = 'BillboardCharts'
table_name_all_artist_appearances = "AllArtistAppearances"
table_name_unique_artist_appearances = "UniqueArtistAppearances"

all_charts_df_together.to_sql(table_name_charts, conn, if_exists='replace', index=True)
total_appearances_all_charts.to_sql(table_name_all_artist_appearances, conn, if_exists='replace', index=False)
unique_chart_appearances.to_sql(table_name_unique_artist_appearances, conn, if_exists='replace', index=False)

701

In [201]:
pd.read_sql("SELECT * from AllArtistAppearances", conn)

,artist,count,week
0,Bad Bunny,180,02-28-2026
1,Olivia Dean,62,02-28-2026
2,sombr,42,02-28-2026
3,Taylor Swift,37,02-28-2026
4,Djo,25,02-28-2026
...,...,...,...
696,Jade LeMac,1,02-28-2026
697,Jamie Fine,1,02-28-2026
698,Jane Zhang,1,02-28-2026
699,Jason Aldean,1,02-28-2026
